In [11]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math
from astropy.io import fits

In [12]:
def distance_calculator(z, zerr, H_0type):
    c = 299792
    H0_mode = H_0type.lower()
    if H0_mode == 'local':
        d_local = c*z/73
        d_localerr = zerr*(c/73)
        return d_local, d_localerr
    if H0_mode == 'cmb':
        d_CMB = c*z/68
        d_CMBerr = zerr*(c/68)
        return d_CMB, d_CMBerr
    else:
        raise ValueError("H_0type must be either 'local' or 'cmb'")

In [13]:
def abs_mag_calculator(m, D_L, D_Lerr): #from distance modulus; given app mag and D_L
    # for M
    M = m - (5*(np.log10(D_L*10**6))) + 5
    # for M_err
    M_err = 2.17147 * (D_Lerr/D_L)
    return M, M_err

In [14]:
def V_band_calculator(color, B): #given B-V and B is given
    V = B - color
    return V

In [15]:
def L_calculator(M, M_err, M_sun): # given that the magnitudes are in the same system
    # for L
    L = 10**((-2/5)*(M - M_sun))
    # for L_err
    L_err = 0.921*L*M_err
    return L, L_err

In [16]:
d_local, d_localerr = distance_calculator(z = 0.001568, zerr = 6.67e-6, H_0type = 'local')
print(d_local, d_localerr)
d_CMB, d_CMBerr = distance_calculator(z = 0.00147, zerr = 1e-05, H_0type = 'cmb')
print(d_CMB, d_CMBerr)

6.439367890410959 0.027391953972602738
6.480797647058823 0.04408705882352942


In [17]:
with fits.open('catalog.fits') as hdul:
    # 2. Access the extension containing the table (usually index 1 or higher)
    # Hint: Use hdul.info() to find which extension has 'BinTableHDU'
    table_hdu = hdul[1] 
    
    # 3. Convert the FITS table data into a pandas DataFrame
    df = pd.DataFrame(table_hdu.data)

# 4. Display the first few rows
print(df.head())

                  objname      pgc   nsaid           specobjid  plate  \
0  2MASXJ01235631+0012308  3519080  114636  783761887031486464    696   
1  2MASXJ04575930-0612323  3689848      -1                  -1     -1   
2  2MASXJ05534289-3205449  3704923      -1                  -1     -1   
3  2MASXJ06094464-3250007  3707868      -1                  -1     -1   
4  2MASXJ07415401+2352188  3723606      -1                  -1     -1   

   fiberid    mjd  group_id          ra        dec  ...  logmass_BV  \
0      493  52209        -1   20.984826   0.208487  ...         NaN   
1       -1     -1        -1   74.497114  -6.208914  ...         NaN   
2       -1     -1        -1   88.428899 -32.095887  ...         NaN   
3       -1     -1        -1   92.436063 -32.833630  ...         NaN   
4       -1     -1        -1  115.475523  23.871764  ...         NaN   

   logmass_BR  logmass  logmass_error  logmass_src  chandra_observation  \
0         NaN    8.166          0.150          g-i         

In [18]:
df.columns

Index(['objname', 'pgc', 'nsaid', 'specobjid', 'plate', 'fiberid', 'mjd',
       'group_id', 'ra', 'dec', 'ra_nsa', 'dec_nsa', 'ra_ned', 'dec_ned',
       'd25', 'v_h', 'v_cmb', 'v_source', 'hl_obj', 'lvg_obj', 'nsa_obj',
       'sga_obj', 't_type', 'color_type', 'best_type', 'a_B_leda', 'a_g_nsa',
       'EBV_irsa', 'Bt0_leda', 'BV_color_leda', 'B_lum', 'gi_color_nsa',
       'i_lum_nsa', 'gr_color_sga', 'r_lum_sga', 'BR_color_ned', 'R_lum_ned',
       'BMag', 'gi_color', 'mag_flag', 'cf3_dist', 'cf3_dist_error',
       'zind_dist', 'zind_dist_error', 'zind_indicator', 'bestdist',
       'bestdist_error', 'bestdist_method', 'bestdist_source', 'dist_ned_flag',
       'logmass_gi', 'logmass_gr', 'logmass_BV', 'logmass_BR', 'logmass',
       'logmass_error', 'logmass_src', 'chandra_observation',
       'chandra_detection', 'log_lx', 'chandra_detection_3arcsec',
       'log_lx_3arcsec'],
      dtype='str')

In [19]:
df = df.copy()

In [20]:
from astropy.table import Table

# 1. Read the FITS table extension using Astropy's native table reader
# (Change hdu=1 if your table is in a different extension index)
astro_table = Table.read('catalog.fits', hdu=1)

# 2. Convert it directly to a pandas DataFrame 
# This automatically handles the Big-Endian to Little-Endian conversion safely
df = astro_table.to_pandas()

# 3. Now the filtering will work flawlessly without any byte-order errors!
NGC1313 = df[df['objname'] == 'NGC1313']

In [21]:
if df['objname'].dtype == object:
    # If it's a byte string, decode it first
    if isinstance(df['objname'].iloc[0], bytes):
        df['objname'] = df['objname'].str.decode('utf-8')
    
    # Strip hidden whitespace padding
    df['objname'] = df['objname'].str.strip()

# Query again
NGC1313 = df[df['objname'] == 'NGC1313']

In [22]:
NGC1313.columns

Index(['objname', 'pgc', 'nsaid', 'specobjid', 'plate', 'fiberid', 'mjd',
       'group_id', 'ra', 'dec', 'ra_nsa', 'dec_nsa', 'ra_ned', 'dec_ned',
       'd25', 'v_h', 'v_cmb', 'v_source', 'hl_obj', 'lvg_obj', 'nsa_obj',
       'sga_obj', 't_type', 'color_type', 'best_type', 'a_B_leda', 'a_g_nsa',
       'EBV_irsa', 'Bt0_leda', 'BV_color_leda', 'B_lum', 'gi_color_nsa',
       'i_lum_nsa', 'gr_color_sga', 'r_lum_sga', 'BR_color_ned', 'R_lum_ned',
       'BMag', 'gi_color', 'mag_flag', 'cf3_dist', 'cf3_dist_error',
       'zind_dist', 'zind_dist_error', 'zind_indicator', 'bestdist',
       'bestdist_error', 'bestdist_method', 'bestdist_source', 'dist_ned_flag',
       'logmass_gi', 'logmass_gr', 'logmass_BV', 'logmass_BR', 'logmass',
       'logmass_error', 'logmass_src', 'chandra_observation',
       'chandra_detection', 'log_lx', 'chandra_detection_3arcsec',
       'log_lx_3arcsec'],
      dtype='str')

In [23]:
NGC1313 = NGC1313[['objname', 'pgc', 'ra', 'dec', 'ra_nsa', 'dec_nsa', 'ra_ned', 'dec_ned',
       'd25', 'v_h', 'v_cmb', 'v_source', 't_type', 'color_type', 'best_type', 'Bt0_leda', 'BV_color_leda', 'B_lum', 'gi_color_nsa',
       'i_lum_nsa', 'gr_color_sga', 'r_lum_sga', 'BR_color_ned', 'R_lum_ned',
       'BMag', 'gi_color', 'mag_flag', 'cf3_dist', 'cf3_dist_error',
       'zind_dist', 'zind_dist_error', 'zind_indicator', 'bestdist',
       'bestdist_error', 'bestdist_method', 'bestdist_source', 'dist_ned_flag']]

In [24]:
ra = NGC1313['ra'].values
dec = NGC1313['dec'].values
print(ra, dec)

[49.5655215] [-66.4978677]


In [25]:
v_h = NGC1313['v_h'].values
v_cmb = NGC1313['v_cmb'].values
print(v_h, v_cmb)

[470.] [651.99]


In [26]:
cf3_dist = NGC1313['cf3_dist'].values
print(cf3_dist)

[3.7896109]


In [27]:
dcmb = 	6.51
dcmb_err = 0.46

In [28]:
cz_cmb = dcmb*68
cz_cmb_err = dcmb_err*68

In [29]:
cz_cmb, cz_cmb_err

(442.68, 31.28)

In [38]:
NGC1313_useful = NGC1313[['Bt0_leda', 'BV_color_leda', 'B_lum', 'd25']]

In [40]:
NGC1313_useful

,Bt0_leda,BV_color_leda,B_lum,d25
3550,9.175,0.395642,8.656105e+09,11.066237


In [41]:
NGC1313_useful.to_csv('NGC1313_useful')

In [31]:
-26.13 - (-26.76)

0.6300000000000026

In [32]:
NGC1313['bestdist'].values

array([4.31], dtype=float32)

In [33]:
M_B, M_Berr = abs_mag_calculator(NGC1313['Bt0_leda'].values[0], NGC1313['bestdist'].values[0], NGC1313['bestdist_error'].values[0])
M_B, M_Berr # abs mag B

(np.float32(-18.997387), np.float32(0.12797062))

In [34]:
m_V = V_band_calculator(NGC1313['BV_color_leda'].values[0], NGC1313['Bt0_leda'].values[0])
m_V # app mag V

np.float32(8.779358)

In [35]:
M_V, M_Verr = abs_mag_calculator(m_V, NGC1313['bestdist'].values[0], NGC1313['bestdist_error'].values[0])
M_V, M_Verr # abs mag V

(np.float32(-19.393028), np.float32(0.12797062))

In [36]:
L_V, L_Verr = L_calculator(M_V, M_Verr,  4.81)
L_V, L_Verr # L V

(np.float32(4.799671e+09), np.float32(5.6569376e+08))

In [3]:
df = pd.read_csv('HECATE_v1.1.csv')
df.head()

,PGC,OBJNAME,ID_NED,ID_NEDD,ID_IRAS,ID_2MASS,SDSS_PHOTID,SDSS_SPECID,RA,DEC,...,FLAG_SFR_HEC,logM_HEC,logSFR_GSW,logM_GSW,MIN_SNR,METAL,FLAG_METAL,CLASS_SP,AGN_S17,AGN_HEC
0,2,UGC12889,UGC 12889,UGC 12889,NaN,00000168+4716282,-,-,0.006945,47.274581,...,NaN,11.0780,NaN,NaN,NaN,NaN,-1,-1,?,?
1,3,PGC000003,MCG -03-01-012,NaN,F23574-1817,00000316-1800289,-,-,0.013125,-18.008200,...,F3,9.5614,NaN,NaN,NaN,NaN,-1,-1,?,?
2,4,PGC000004,KUG 2357+228,KUG 2357+228,NaN,NaN,1237679478544072744,-,0.014292,23.087596,...,NaN,NaN,NaN,NaN,NaN,NaN,-1,-1,?,?
3,5,IC5370,IC 5370,NaN,NaN,00000914+3244182,1237663234987458745,-,0.038121,32.738420,...,W3,11.3953,NaN,NaN,NaN,NaN,-1,-1,?,?
4,6,PGC000006,KUG 2357+156,NaN,F23574+1535,00000214+1552539,1237656496724312079,844551136588359680,0.008763,15.881670,...,F3,10.1894,-0.239,9.94,21.2,8.7275,0,0,?,N


In [5]:
df_f = df[df['OBJNAME'] == 'NGC1313'] 

In [8]:
df_f.columns

Index(['PGC', 'OBJNAME', 'ID_NED', 'ID_NEDD', 'ID_IRAS', 'ID_2MASS',
       'SDSS_PHOTID', 'SDSS_SPECID', 'RA', 'DEC', 'F_ASTROM', 'R1', 'R2', 'PA',
       'RSOURCE', 'RFLAG', 'T', 'E_T', 'INCL', 'V', 'E_V', 'V_VIR', 'E_V_VIR',
       'NDIST', 'EDIST', 'D', 'E_D', 'D_LO68', 'D_HI68', 'D_LO95', 'D_HI95',
       'DMETHOD', 'UT', 'BT', 'VT', 'IT', 'E_UT', 'E_BT', 'E_VT', 'E_IT', 'AG',
       'AI', 'S12', 'S25', 'S60', 'S100', 'Q12', 'Q25', 'Q60', 'Q100', 'WF1',
       'WF2', 'WF3', 'WF4', 'E_WF1', 'E_WF2', 'E_WF3', 'E_WF4', 'WFPOINT',
       'WFTREAT', 'J', 'H', 'K', 'E_J', 'E_H', 'E_K', 'FLAG_2MASS', 'U', 'G',
       'R', 'I', 'Z', 'E_U', 'E_G', 'E_R', 'E_I', 'E_Z', 'logL_TIR',
       'logL_FIR', 'logL_60u', 'logL_12u', 'logL_22u', 'logL_K', 'ML_RATIO',
       'logSFR_TIR', 'logSFR_FIR', 'logSFR_60u', 'logSFR_12u', 'logSFR_22u',
       'logSFR_HEC', 'FLAG_SFR_HEC', 'logM_HEC', 'logSFR_GSW', 'logM_GSW',
       'MIN_SNR', 'METAL', 'FLAG_METAL', 'CLASS_SP', 'AGN_S17', 'AGN_HEC'],
      dtyp

In [10]:
df_f = df_f[['RA', 'DEC', 'BT', 'VT', 'E_BT', 'E_VT']]
df_f

,RA,DEC,BT,VT,E_BT,E_VT
8468,49.565522,-66.497868,9.648,9.487,0.414,0.11
